# SageMaker V3 Distributed Local Training Example

This notebook demonstrates how to run distributed training locally using SageMaker V3 ModelTrainer with multiple Docker containers. Note: This notebook will not run in SageMaker Studio. 

In [1]:
import os
import subprocess
import tempfile
import shutil
import numpy as np

from sagemaker.train.model_trainer import ModelTrainer, Mode
from sagemaker.train.configs import SourceCode, Compute, InputData
from sagemaker.train.distributed import Torchrun
from sagemaker.core.helper.session_helper import Session
from sagemaker.core import image_uris

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /tmp/sm_config_1024.yaml


In [2]:
# NOTE: Local mode requires Docker to be installed and running.
import os
os.environ['PATH'] = '/usr/local/bin:/Applications/Docker.app/Contents/Resources/bin:' + os.environ['PATH']

## Step 1: Setup Session and Create Test Data

Initialize the SageMaker session and create the necessary test data and training script.

In [3]:
sagemaker_session = Session()
region = sagemaker_session.boto_region_name

DEFAULT_CPU_IMAGE = image_uris.retrieve(
    framework="pytorch",
    region=region,
    version="2.0.0",
    py_version="py310",
    instance_type="ml.m5.xlarge",
    image_scope="training"
)

# Create temporary directories
temp_dir = tempfile.mkdtemp()
source_dir = os.path.join(temp_dir, "source")
data_dir = os.path.join(temp_dir, "data")
train_dir = os.path.join(data_dir, "train")
test_dir = os.path.join(data_dir, "test")

os.makedirs(source_dir, exist_ok=True)
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

print(f"Created temporary directories in: {temp_dir}")
print("Note: This will use multiple Docker containers locally for distributed training!")

Created temporary directories in: /tmp/tmpqfaekn48
Note: This will use multiple Docker containers locally for distributed training!


## Step 2: Create Training Data and Scripts

Generate the test data and training scripts needed for distributed local training.

In [4]:
# Create test data
np.random.seed(42)
x_train = np.random.randn(100, 4).astype(np.float32)
y_train = np.random.randn(100).astype(np.float32)
x_test = np.random.randn(20, 4).astype(np.float32)
y_test = np.random.randn(20).astype(np.float32)

np.save(os.path.join(train_dir, "x_train.npy"), x_train)
np.save(os.path.join(train_dir, "y_train.npy"), y_train)
np.save(os.path.join(test_dir, "x_test.npy"), x_test)
np.save(os.path.join(test_dir, "y_test.npy"), y_test)

print(f"Created training data: {x_train.shape}, {y_train.shape}")
print(f"Created test data: {x_test.shape}, {y_test.shape}")

Created training data: (100, 4), (100,)
Created test data: (20, 4), (20,)


In [5]:
# Create pytorch model definition
pytorch_model_def = '''
import torch
import torch.nn as nn

def get_model():
    return nn.Sequential(
        nn.Linear(4, 10),
        nn.ReLU(),
        nn.Linear(10, 1)
    )
'''

with open(os.path.join(source_dir, "pytorch_model_def.py"), 'w') as f:
    f.write(pytorch_model_def)

print("Created pytorch_model_def.py")

Created pytorch_model_def.py


In [6]:
# Create training script (same as single container for simplicity)
training_script = '''
import argparse
import numpy as np
import os
import sys
import logging
import json
import shutil
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pytorch_model_def import get_model

logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
logger.addHandler(logging.StreamHandler(sys.stdout))
current_dir = os.path.dirname(os.path.abspath(__file__))
data_dir = "/opt/ml/input/data"

def get_train_data(train_dir):
    x_train = np.load(os.path.join(train_dir, "x_train.npy"))
    y_train = np.load(os.path.join(train_dir, "y_train.npy"))
    logger.info(f"x train: {x_train.shape}, y train: {y_train.shape}")
    return torch.from_numpy(x_train), torch.from_numpy(y_train)

def get_test_data(test_dir):
    x_test = np.load(os.path.join(test_dir, "x_test.npy"))
    y_test = np.load(os.path.join(test_dir, "y_test.npy"))
    logger.info(f"x test: {x_test.shape}, y test: {y_test.shape}")
    return torch.from_numpy(x_test), torch.from_numpy(y_test)

def train():
    train_dir = os.path.join(data_dir, "train")
    test_dir = os.path.join(data_dir, "test")
    model_dir = os.environ.get("SM_MODEL_DIR", os.path.join(current_dir, "data/model"))

    x_train, y_train = get_train_data(train_dir)
    x_test, y_test = get_test_data(test_dir)
    train_ds = TensorDataset(x_train, y_train)

    batch_size = 64
    epochs = 1
    learning_rate = 0.1
    logger.info(f"batch_size = {batch_size}, epochs = {epochs}, learning rate = {learning_rate}")

    train_dl = DataLoader(train_ds, batch_size, shuffle=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = get_model().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    for epoch in range(epochs):
        for x_train_batch, y_train_batch in train_dl:
            y = model(x_train_batch.float())
            loss = criterion(y.flatten(), y_train_batch.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        epoch += 1
        logger.info(f"epoch: {epoch} -> loss: {loss}")

    with torch.no_grad():
        y = model(x_test.float()).flatten()
        mse = ((y - y_test) ** 2).sum() / y_test.shape[0]
    print("Test MSE:", mse.numpy())

    os.makedirs(model_dir, exist_ok=True)
    torch.save(model.state_dict(), model_dir + "/model.pth")
    
    inference_code_path = model_dir + "/code/"
    if not os.path.exists(inference_code_path):
        os.mkdir(inference_code_path)
        logger.info(f"Created a folder at {inference_code_path}!")

    shutil.copy("local_training_script.py", inference_code_path)
    shutil.copy("pytorch_model_def.py", inference_code_path)
    logger.info(f"Saving models files to {inference_code_path}")

if __name__ == "__main__":
    print("Running the training job ...")
    train()
'''

with open(os.path.join(source_dir, "local_training_script.py"), 'w') as f:
    f.write(training_script)

print("Created local_training_script.py")

Created local_training_script.py


## Step 3: Configure Distributed Local Training

Set up ModelTrainer for distributed training in local container mode.

In [7]:
source_code = SourceCode(
    source_dir=source_dir,
    entry_script="local_training_script.py",
)

distributed = Torchrun(
    process_count_per_node=1,
)

compute = Compute(
    instance_type="local_cpu",
    instance_count=2,
)

train_data = InputData(
    channel_name="train",
    data_source=train_dir,
)

test_data = InputData(
    channel_name="test",
    data_source=test_dir,
)

print("Distributed Local Training Configuration:")
print(f"  Containers: {compute.instance_count}")
print(f"  Processes per container: {distributed.process_count_per_node}")
print(f"  Total processes: {compute.instance_count * distributed.process_count_per_node}")
print(f"  Distributed framework: Torchrun")

Distributed Local Training Configuration:
  Containers: 2
  Processes per container: 1
  Total processes: 2
  Distributed framework: Torchrun


## Step 4: Create Distributed ModelTrainer

Initialize ModelTrainer for distributed local container training.

In [8]:
model_trainer = ModelTrainer(
    training_image=DEFAULT_CPU_IMAGE,
    sagemaker_session=sagemaker_session,
    source_code=source_code,
    distributed=distributed,
    compute=compute,
    input_data_config=[train_data, test_data],
    base_job_name="local_mode_multi_container",
    training_mode=Mode.LOCAL_CONTAINER,
)

print("Distributed ModelTrainer created successfully!")
print(f"Training mode: {Mode.LOCAL_CONTAINER}")
print(f"Distributed config: {distributed}")

[07/14/26 07:09:11] INFO     StoppingCondition not provided. Using default:                         ]8;id=2297719;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=2297720;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     Training image URI:                                               ]8;id=2297727;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=2297728;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.0                     
                             .0-cpu-py310                                                                          

Distributed ModelTrainer created successfully!
Training mode: Mode.LOCAL_CONTAINER
Distributed config: process_count_per_node=1 smp=None


## Step 5: Run Distributed Local Training

Start the distributed training job using multiple local containers.

In [9]:
print("Starting distributed local container training...")
print("This will launch 2 Docker containers with 1 training process each.")

try:
    model_trainer.train()
    print("Distributed local container training completed successfully!")
    operation_successful = True
except Exception as e:
    print(f"Training failed with error: {e}")
    operation_successful = False

Starting distributed local container training...
This will launch 2 Docker containers with 1 training process each.


                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=2297735;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=2297736;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#287\287]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

Login Succeeded



WARNING! Your credentials are stored unencrypted in '/root/.docker/config.json'.
Configure a credential helper to remove this warning. See
https://docs.docker.com/go/credential-store/



[07/14/26 07:09:12] INFO     docker command: docker pull                                              ]8;id=2297743;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py\image.py]8;;\:]8;id=2297744;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/core/local/image.py#1242\1242]8;;\
                             763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.0.0-cpu-              
                             py310                                                                                 

...

algo-2-1  | Running the training job ...
algo-2-1  | x train: (100, 4), y train: (100,)
algo-2-1  | x test: (20, 4), y test: (20,)
algo-2-1  | batch_size = 64, epochs = 1, learning rate = 0.1
algo-2-1  | epoch: 1 -> loss: 1.0963761806488037
algo-1-1  | Running the training job ...
algo-2-1  | Test MSE: 0.6335511
algo-2-1  | Created a folder at /opt/ml/model/code/!
algo-1-1  | x train: (100, 4), y train: (100,)
algo-2-1  | Saving models files to /opt/ml/model/code/
algo-1-1  | x test: (20, 4), y test: (20,)
algo-1-1  | batch_size = 64, epochs = 1, learning rate = 0.1
algo-1-1  | epoch: 1 -> loss: 1.0892539024353027
algo-1-1  | Test MSE: 0.78975
algo-1-1  | Saving models files to /opt/ml/model/code/


algo-2-1  | Training Container Execution Completed
algo-2-1  | ++ echo 'Training Container Execution Completed'
algo-1-1  | ++ echo 'Training Container Execution Completed'
algo-1-1  | Training Container Execution Completed


algo-2-1 exited with code 0
 Compose Stopping Aborting on container exit...
 Container training-examples-algo-1-1 Stopping 
 Container training-examples-algo-2-1 Stopping 
 Container training-examples-algo-2-1 Stopped 
 Container training-examples-algo-1-1 Stopped 
algo-1-1 exited with code 0


[07/14/26 07:10:47] INFO     Local training job completed, output artifacts saved to         ]8;id=2297785;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/train/local/local_container.py\local_container.py]8;;\:]8;id=2297786;file:///root/.pyenv/versions/3.12.13/lib/python3.12/site-packages/sagemaker/train/local/local_container.py#248\248]8;;\
                             file:///codebuild/output/src3861286305/src/sagemaker-python-sdk                       
                             /v3-examples/training-examples/compressed_artifacts/model.tar.g                       
                             z                                                                                     

Distributed local container training completed successfully!


## Step 6: Check Training Results

Examine the results from distributed training.

In [10]:
if operation_successful:
    current_dir = os.getcwd()
    
    print("Distributed Training Results:")
    print("=" * 35)
    
    # Check that certain directories don't exist (cleanup verification)
    assert not os.path.exists(os.path.join(current_dir, "shared"))
    assert not os.path.exists(os.path.join(current_dir, "input"))
    assert not os.path.exists(os.path.join(current_dir, "algo-1"))
    assert not os.path.exists(os.path.join(current_dir, "algo-2"))
    print("✓ Temporary directories properly cleaned up")
    
    # Check for expected artifacts
    directories_to_check = [
        "compressed_artifacts",
        "artifacts",
        "model",
        "output",
    ]
    
    for directory in directories_to_check:
        path = os.path.join(current_dir, directory)
        if os.path.exists(path):
            print(f"✓ {directory}: Found")
        else:
            print(f"✗ {directory}: Not found")
    
    print("\nDistributed Training Configuration:")
    print(f"  Training Image: {DEFAULT_CPU_IMAGE}")
    print(f"  Container Count: {compute.instance_count}")
    print(f"  Processes per Container: {distributed.process_count_per_node}")
    print(f"  Total Training Processes: {compute.instance_count * distributed.process_count_per_node}")
    
else:
    print("Training was not successful.")

Distributed Training Results:
✓ Temporary directories properly cleaned up
✓ compressed_artifacts: Found
✓ artifacts: Found
✓ model: Found
✓ output: Found

Distributed Training Configuration:
  Training Image: 763104351884.dkr.ecr.us-west-2.amazonaws.com/pytorch-training:2.0.0-cpu-py310
  Container Count: 2
  Processes per Container: 1
  Total Training Processes: 2


## Step 7: Clean Up

Clean up local artifacts and temporary files.

In [11]:
try:
    subprocess.run(["docker", "compose", "down", "-v"], check=False)
    print("Docker containers stopped")
except Exception:
    pass

# Clean up temporary files
try:
    shutil.rmtree(temp_dir)
    print(f"Cleaned up temporary directory: {temp_dir}")
except Exception as e:
    print(f"Could not clean up temp directory: {e}")

# Clean up training artifacts
current_dir = os.getcwd()
directories = ["compressed_artifacts", "artifacts", "model", "output"]

for directory in directories:
    path = os.path.join(current_dir, directory)
    if os.path.exists(path):
        try:
            shutil.rmtree(path)
            print(f"Cleaned up: {directory}")
        except Exception as e:
            print(f"Could not clean up {directory}: {e}")

# Final assertion
assert operation_successful
print("\n✓ Distributed local training completed successfully!")
print("Cleanup completed - all artifacts removed.")

 Container training-examples-algo-1-1 Stopping 
 Container training-examples-algo-2-1 Stopping 
 Container training-examples-algo-1-1 Stopped 
 Container training-examples-algo-1-1 Removing 
 Container training-examples-algo-2-1 Stopped 
 Container training-examples-algo-2-1 Removing 
 Container training-examples-algo-1-1 Removed 
 Container training-examples-algo-2-1 Removed 
 Network sagemaker-local Removing 


Docker containers stopped
Cleaned up temporary directory: /tmp/tmpqfaekn48
Cleaned up: compressed_artifacts
Cleaned up: artifacts
Cleaned up: model
Cleaned up: output

✓ Distributed local training completed successfully!
Cleanup completed - all artifacts removed.


 Network sagemaker-local Removed 


## Summary

This notebook demonstrated:
1. **Multi-container distributed training**: Running training across multiple Docker containers locally
2. **Torchrun integration**: Using SageMaker's Torchrun distributed driver
3. **Local development workflow**: Testing distributed training before cloud deployment
4. **Proper cleanup**: Following cleanup patterns for local artifacts

Distributed local training provides a great way to test distributed training patterns locally before deploying to SageMaker cloud instances, with no AWS costs and realistic container-based execution.